In [1]:
import polars as pl
import pandas as pd
import time

# Считываем данные с excel. Ручной режим

In [2]:
FILE_PATH = "лекция 1/ШЧ_2_corr.xlsx"

# 1. Определение количества листов
xls = pd.ExcelFile(FILE_PATH, engine="openpyxl")
sheet_names = xls.sheet_names
n_sheets = len(sheet_names)
print(f"Количество листов: {n_sheets}")
print(f"Названия листов: {sheet_names}")


Количество листов: 6
Названия листов: ['2019', '2018', '2017', '2016', '2015', '2014']


In [4]:
# Считывание каждого листа отдельно + объединение в 1 массив
df_2019 = pl.read_excel(FILE_PATH, sheet_name='2019')
# Добавим новый столбец со значение-константой: годом события
# способ добавить столбец в Polars (pl.lit(sheet) — создание литерала (константы), .alias("Год_листа") — присвоение имени)
df_2019 = df_2019.with_columns(
        pl.lit(2019).alias("Год_листа")
    )
df_2018 = pl.read_excel(FILE_PATH, sheet_name='2018')
df_2018 = df_2018.with_columns(
        pl.lit(2018).alias("Год_листа")
    )
df_2017 = pl.read_excel(FILE_PATH, sheet_name='2017')
df_2017 = df_2017.with_columns(
        pl.lit(2017).alias("Год_листа")
    )
df_2016 = pl.read_excel(FILE_PATH, sheet_name='2016')
df_2016 = df_2016.with_columns(
        pl.lit(2016).alias("Год_листа")
    )
df_2015 = pl.read_excel(FILE_PATH, sheet_name='2015')
df_2015 = df_2015.with_columns(
        pl.lit(2015).alias("Год_листа")
    )
df_2014 = pl.read_excel(FILE_PATH, sheet_name='2014')
df_2014 = df_2014.with_columns(
        pl.lit(2014).alias("Год_листа")
    )
result = pl.concat([df_2019, df_2018, df_2017, df_2016, df_2015, df_2014], how="diagonal_relaxed")
result[:1]


Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string


__UNNAMED__0,Дата осмотра,Осмотр проводил,"Станция, перегон","Объекты, количество отступлений",Классификация (Группа),Классификация (Вид),Примечание,Крайний срок,Статус,Причина,Дата устранения,Год_листа
i64,date,str,str,str,str,str,str,date,str,str,date,i32
0,2019-04-03,"""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №39 устранить люфт дли…",2019-04-13,"""ЗАКРЫТО""",null,2019-04-11,2019


In [ ]:
# ЗАДАНИЕ № 1. Посчитайте количество инцидентов по годам, а так же общее количество инцидентов за весь период наблюдения. 
# Придумайте, как сделать проверку правильности реализации кода

# Считываем данные с excel. Автоматизация

In [ ]:
# Объявляем список,
dfs = []
for sheet in sheet_names:    
    df = pl.read_excel(FILE_PATH, sheet_name=sheet)    
    # способ добавить столбец в Polars (pl.lit(sheet) — создание литерала (константы), .alias("Год_листа") — присвоение имени)
    df = df.with_columns(
        pl.lit(sheet).alias("Год_листа")
    )
    dfs.append(df)
    # объединяем все датафреймы в один (диагонально - на случай разных наборов колонок)
result = pl.concat(dfs, how="diagonal_relaxed")

Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string


In [ ]:
# ЗАДАНИЕ № 2 посчитайте теже показатели, что в задании 1. Сравните результаты.
# ЗАДАНИЕ № 2, дополннение (для уровня +). Оберните код выше в функцию, рассчитайте время вполнения.

# Генерация признаков

In [ ]:
result[:1]

__UNNAMED__0,Дата осмотра,Осмотр проводил,"Станция, перегон","Объекты, количество отступлений",Классификация (Группа),Классификация (Вид),Примечание,Крайний срок,Статус,Причина,Дата устранения,Год_листа
i64,date,str,str,str,str,str,str,date,str,str,date,str
0,2019-04-03,"""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №39 устранить люфт дли…",2019-04-13,"""ЗАКРЫТО""",null,2019-04-11,"""2019"""


In [9]:
# Сделаем копию нашего массива. ВОПРОС: для чего?
df = result.clone()
# Приводим колонку "Дата осмотра" к типу datetime, если она строковая
df = df.with_columns(
    pl.col("Дата осмотра").str.strptime(pl.Datetime, strict=False, format=None)
    if df.schema["Дата осмотра"] == pl.Utf8
    else pl.col("Дата осмотра")
)

# ===========================================================
# 5.1 Извлечение признаков из даты
# ===========================================================
df = df.with_columns([
    pl.col("Дата осмотра").dt.week().alias("Номер_недели"),      # номер недели в году
    pl.col("Дата осмотра").dt.ordinal_day().alias("Номер_дня"),  # номер дня в году
    pl.col("Дата осмотра").dt.month().alias("Номер_месяца"),     # номер месяца
    pl.col("Дата осмотра").dt.year().alias("Год"),                # год
])

print("\nПример новых признаков даты:")
print(df.select(["Дата осмотра", "Год", "Номер_месяца", "Номер_недели", "Номер_дня"]).head(10))


Пример новых признаков даты:
shape: (10, 5)
┌──────────────┬──────┬──────────────┬──────────────┬───────────┐
│ Дата осмотра ┆ Год  ┆ Номер_месяца ┆ Номер_недели ┆ Номер_дня │
│ ---          ┆ ---  ┆ ---          ┆ ---          ┆ ---       │
│ date         ┆ i32  ┆ i8           ┆ i8           ┆ i16       │
╞══════════════╪══════╪══════════════╪══════════════╪═══════════╡
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4      

In [10]:
# Группировка: количество событий по неделям и по годам

# Количество событий по неделям (в разрезе год+неделя)
events_by_week = (
    df.group_by(["Год", "Номер_недели"])
    .agg(pl.count().alias("Количество_событий"))
    .sort(["Год", "Номер_недели"])
)
print("\nКоличество событий по неделям:")
print(events_by_week)

# Количество событий по годам
events_by_year = (
    df.group_by("Год")
    .agg(pl.count().alias("Количество_событий"))
    .sort("Год")
)
print("\nКоличество событий по годам:")
print(events_by_year)


Количество событий по неделям:
shape: (296, 3)
┌──────┬──────────────┬────────────────────┐
│ Год  ┆ Номер_недели ┆ Количество_событий │
│ ---  ┆ ---          ┆ ---                │
│ i32  ┆ i8           ┆ u32                │
╞══════╪══════════════╪════════════════════╡
│ 2014 ┆ 13           ┆ 1                  │
│ 2014 ┆ 14           ┆ 70                 │
│ 2014 ┆ 15           ┆ 99                 │
│ 2014 ┆ 16           ┆ 135                │
│ 2014 ┆ 17           ┆ 67                 │
│ …    ┆ …            ┆ …                  │
│ 2019 ┆ 48           ┆ 138                │
│ 2019 ┆ 49           ┆ 26                 │
│ 2019 ┆ 50           ┆ 95                 │
│ 2019 ┆ 51           ┆ 120                │
│ 2019 ┆ 52           ┆ 106                │
└──────┴──────────────┴────────────────────┘

Количество событий по годам:
shape: (6, 2)
┌──────┬────────────────────┐
│ Год  ┆ Количество_событий │
│ ---  ┆ ---                │
│ i32  ┆ u32                │
╞══════╪═══════════════

C:\Users\oesune\AppData\Local\Temp\ipykernel_20352\1533297078.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("Количество_событий"))
C:\Users\oesune\AppData\Local\Temp\ipykernel_20352\1533297078.py:15: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("Количество_событий"))


# Переход к задаче ML

In [13]:
df = df.with_columns([
    pl.col("Крайний срок").str.strptime(pl.Datetime, strict=False, format=None)
        if df.schema["Крайний срок"] == pl.Utf8 else pl.col("Крайний срок"),
    pl.col("Дата устранения").str.strptime(pl.Datetime, strict=False, format=None)
        if df.schema["Дата устранения"] == pl.Utf8 else pl.col("Дата устранения"),
])

# Формирование целевой переменной (target)
df = df.with_columns(
    pl.when(pl.col("Статус").is_null().is_null())
      .then(None)
      .when(pl.col("Статус") =='ЗАКРЫТО')
      .then(1)
      .when(pl.col("Статус") =='УСТРАНЕН')
      .then(0)
      .otherwise(-1)
      .alias("Статус_устранения")
)


In [14]:
# Задание 3. Проверьте скольок раз инцидент был устранен во время, а сколько - просрочне.
df['Статус_устранения'].value_counts()

Статус_устранения,count
i32,u32
0,8298
1,37691
-1,2


In [ ]:
# Дополнительные признаки, полезные для модели 
df = df.with_columns([
    # срок на устранение в днях (от даты осмотра до крайнего срока)
    (pl.col("Крайний срок") - pl.col("Дата осмотра")).dt.total_days().alias("Срок_на_устранение_дней"),

    # день недели осмотра (0 = понедельник, 6 = воскресенье)
    pl.col("Дата осмотра").dt.weekday().alias("День_недели_осмотра"),

    # признак: есть ли примечание (текстовый комментарий) у нарушения
    pl.col("Примечание").is_not_null().cast(pl.Int8).alias("Есть_примечание"),

    # длина текста примечания (если есть) как показатель "сложности" нарушения
    pl.col("Примечание").fill_null("").str.len_chars().alias("Длина_примечания"),
])

# Удаляем строки, где target не определён (нет обеих дат) ---
df_model = df.filter(pl.col("Статус_устранения").is_not_null())

print(f"Строк для обучения (с известным target): {df_model.shape[0]} из {df.shape[0]}")

# --- 6.5 Кодирование категориальных признаков ---
# Для классических ML-моделей (sklearn, xgboost) категории нужно закодировать в числа
categorical_cols = [
    "Станция, перегон",
    "Классификация (Группа)",
    "Классификация (Вид)",
    "Осмотр проводил",
]
# Приводим качественный признаки к числовому виду методом горячего кодирования 
# Метод .to_dummies() создаёт столбцы по шаблону {имя_столбца}_{значение}
# например:

# Станция, перегон_станция_33
# Станция, перегон_станция_96
# Классификация (Группа)_группа_49
# Осмотр проводил_Сотрудник_375
df_model = df_model.to_dummies(columns=categorical_cols)
df_model[:1]



# Альтернативный метод
# # Простое кодирование через to_physical() поверх Categorical-типа
# df_model = df_model.with_columns([
#     pl.col(col).cast(pl.Categorical).to_physical().alias(f"{col}_encoded")
#     for col in categorical_cols
# ])
# 1. pl.col(col).cast(pl.Categorical)
# Преобразует текстовый столбец (Utf8) в тип Categorical. Внутри Polars это работает так:

#Каждому уникальному текстовому значению присваивается числовой код (целое число).
#Строится словарь соответствия «код ↔ строка» (так называемый string cache / категориальная таблица).
#Физически столбец хранится как массив чисел + таблица расшифровки, а не как массив строк — это экономит память и ускоряет операции сравнения/группировки.

# 2. .to_physical()
# Извлекает именно числовое представление (физический слой) категориального столбца — то есть просто коды без сохранения связи со строковыми значениями. Результат — обычный столбец типа UInt32 (или UInt8/UInt16/UInt32 в зависимости от количества уникальных значений).
#ВАЖНО!!! АВТОМАТИЧЕСКИ НЕ СОХРАНЯЕТСЯ СООТВЕСТВИЕ ЧИСЛА И КОДА!


Строк для обучения (с известным target): 45991 из 45991


__UNNAMED__0,Дата осмотра,Осмотр проводил_Сотрудник_0,Осмотр проводил_Сотрудник_1,Осмотр проводил_Сотрудник_10,Осмотр проводил_Сотрудник_100,Осмотр проводил_Сотрудник_101,Осмотр проводил_Сотрудник_102,Осмотр проводил_Сотрудник_103,Осмотр проводил_Сотрудник_104,Осмотр проводил_Сотрудник_105,Осмотр проводил_Сотрудник_106,Осмотр проводил_Сотрудник_107,Осмотр проводил_Сотрудник_108,Осмотр проводил_Сотрудник_109,Осмотр проводил_Сотрудник_11,Осмотр проводил_Сотрудник_110,Осмотр проводил_Сотрудник_111,Осмотр проводил_Сотрудник_112,Осмотр проводил_Сотрудник_113,Осмотр проводил_Сотрудник_114,Осмотр проводил_Сотрудник_115,Осмотр проводил_Сотрудник_116,Осмотр проводил_Сотрудник_117,Осмотр проводил_Сотрудник_118,Осмотр проводил_Сотрудник_119,Осмотр проводил_Сотрудник_120,Осмотр проводил_Сотрудник_121,Осмотр проводил_Сотрудник_122,Осмотр проводил_Сотрудник_123,Осмотр проводил_Сотрудник_124,Осмотр проводил_Сотрудник_125,Осмотр проводил_Сотрудник_126,Осмотр проводил_Сотрудник_127,Осмотр проводил_Сотрудник_128,Осмотр проводил_Сотрудник_129,Осмотр проводил_Сотрудник_13,…,Классификация (Вид)_вид_80,Классификация (Вид)_вид_81,Классификация (Вид)_вид_82,Классификация (Вид)_вид_83,Классификация (Вид)_вид_84,Классификация (Вид)_вид_85,Классификация (Вид)_вид_86,Классификация (Вид)_вид_87,Классификация (Вид)_вид_88,Классификация (Вид)_вид_89,Классификация (Вид)_вид_9,Классификация (Вид)_вид_90,Классификация (Вид)_вид_91,Классификация (Вид)_вид_92,Классификация (Вид)_вид_93,Классификация (Вид)_вид_94,Классификация (Вид)_вид_95,Классификация (Вид)_вид_96,Классификация (Вид)_вид_97,Классификация (Вид)_вид_98,Классификация (Вид)_вид_99,Примечание,Крайний срок,Статус,Причина,Дата устранения,Год_листа,Номер_недели,Номер_дня,Номер_месяца,Год,target_просрочено,Статус_устранения,Срок_на_устранение_дней,День_недели_осмотра,Есть_примечание,Длина_примечания
i64,date,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,str,date,str,str,date,str,i8,i16,i8,i32,i32,i32,i64,i8,i8,u32
0,2019-04-03,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Стрелка №39 устранить люфт дли…",2019-04-13,"""ЗАКРЫТО""",null,2019-04-11,"""2019""",14,93,4,2019,0,1,10,3,1,52


In [28]:
# все столбцы, порождённые one-hot кодированием (по префиксу исходных названий)
onehot_cols = [
    col for col in df_model.columns
    if any(col.startswith(cat_col) for cat_col in categorical_cols)
]
df_model[onehot_cols]

Осмотр проводил_Сотрудник_0,Осмотр проводил_Сотрудник_1,Осмотр проводил_Сотрудник_10,Осмотр проводил_Сотрудник_100,Осмотр проводил_Сотрудник_101,Осмотр проводил_Сотрудник_102,Осмотр проводил_Сотрудник_103,Осмотр проводил_Сотрудник_104,Осмотр проводил_Сотрудник_105,Осмотр проводил_Сотрудник_106,Осмотр проводил_Сотрудник_107,Осмотр проводил_Сотрудник_108,Осмотр проводил_Сотрудник_109,Осмотр проводил_Сотрудник_11,Осмотр проводил_Сотрудник_110,Осмотр проводил_Сотрудник_111,Осмотр проводил_Сотрудник_112,Осмотр проводил_Сотрудник_113,Осмотр проводил_Сотрудник_114,Осмотр проводил_Сотрудник_115,Осмотр проводил_Сотрудник_116,Осмотр проводил_Сотрудник_117,Осмотр проводил_Сотрудник_118,Осмотр проводил_Сотрудник_119,Осмотр проводил_Сотрудник_120,Осмотр проводил_Сотрудник_121,Осмотр проводил_Сотрудник_122,Осмотр проводил_Сотрудник_123,Осмотр проводил_Сотрудник_124,Осмотр проводил_Сотрудник_125,Осмотр проводил_Сотрудник_126,Осмотр проводил_Сотрудник_127,Осмотр проводил_Сотрудник_128,Осмотр проводил_Сотрудник_129,Осмотр проводил_Сотрудник_13,Осмотр проводил_Сотрудник_130,Осмотр проводил_Сотрудник_131,…,Классификация (Вид)_вид_66,Классификация (Вид)_вид_67,Классификация (Вид)_вид_68,Классификация (Вид)_вид_69,Классификация (Вид)_вид_7,Классификация (Вид)_вид_70,Классификация (Вид)_вид_71,Классификация (Вид)_вид_72,Классификация (Вид)_вид_73,Классификация (Вид)_вид_74,Классификация (Вид)_вид_75,Классификация (Вид)_вид_76,Классификация (Вид)_вид_77,Классификация (Вид)_вид_78,Классификация (Вид)_вид_79,Классификация (Вид)_вид_8,Классификация (Вид)_вид_80,Классификация (Вид)_вид_81,Классификация (Вид)_вид_82,Классификация (Вид)_вид_83,Классификация (Вид)_вид_84,Классификация (Вид)_вид_85,Классификация (Вид)_вид_86,Классификация (Вид)_вид_87,Классификация (Вид)_вид_88,Классификация (Вид)_вид_89,Классификация (Вид)_вид_9,Классификация (Вид)_вид_90,Классификация (Вид)_вид_91,Классификация (Вид)_вид_92,Классификация (Вид)_вид_93,Классификация (Вид)_вид_94,Классификация (Вид)_вид_95,Классификация (Вид)_вид_96,Классификация (Вид)_вид_97,Классификация (Вид)_вид_98,Классификация (Вид)_вид_99
u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [33]:
#  Формирование финального набора признаков X и цели y
# базовые числовые признаки (не связанные с категориями)
numeric_feature_cols = [
    "Год", "Номер_месяца", "Номер_недели", "Номер_дня", "День_недели_осмотра",
    "Срок_на_устранение_дней", "Есть_примечание", "Длина_примечания",
]
fea_col = numeric_feature_cols + onehot_cols
df_fea = df_model[fea_col]
df_fea[:1]



Год,Номер_месяца,Номер_недели,Номер_дня,День_недели_осмотра,Срок_на_устранение_дней,Есть_примечание,Длина_примечания,Осмотр проводил_Сотрудник_0,Осмотр проводил_Сотрудник_1,Осмотр проводил_Сотрудник_10,Осмотр проводил_Сотрудник_100,Осмотр проводил_Сотрудник_101,Осмотр проводил_Сотрудник_102,Осмотр проводил_Сотрудник_103,Осмотр проводил_Сотрудник_104,Осмотр проводил_Сотрудник_105,Осмотр проводил_Сотрудник_106,Осмотр проводил_Сотрудник_107,Осмотр проводил_Сотрудник_108,Осмотр проводил_Сотрудник_109,Осмотр проводил_Сотрудник_11,Осмотр проводил_Сотрудник_110,Осмотр проводил_Сотрудник_111,Осмотр проводил_Сотрудник_112,Осмотр проводил_Сотрудник_113,Осмотр проводил_Сотрудник_114,Осмотр проводил_Сотрудник_115,Осмотр проводил_Сотрудник_116,Осмотр проводил_Сотрудник_117,Осмотр проводил_Сотрудник_118,Осмотр проводил_Сотрудник_119,Осмотр проводил_Сотрудник_120,Осмотр проводил_Сотрудник_121,Осмотр проводил_Сотрудник_122,Осмотр проводил_Сотрудник_123,Осмотр проводил_Сотрудник_124,…,Классификация (Вид)_вид_66,Классификация (Вид)_вид_67,Классификация (Вид)_вид_68,Классификация (Вид)_вид_69,Классификация (Вид)_вид_7,Классификация (Вид)_вид_70,Классификация (Вид)_вид_71,Классификация (Вид)_вид_72,Классификация (Вид)_вид_73,Классификация (Вид)_вид_74,Классификация (Вид)_вид_75,Классификация (Вид)_вид_76,Классификация (Вид)_вид_77,Классификация (Вид)_вид_78,Классификация (Вид)_вид_79,Классификация (Вид)_вид_8,Классификация (Вид)_вид_80,Классификация (Вид)_вид_81,Классификация (Вид)_вид_82,Классификация (Вид)_вид_83,Классификация (Вид)_вид_84,Классификация (Вид)_вид_85,Классификация (Вид)_вид_86,Классификация (Вид)_вид_87,Классификация (Вид)_вид_88,Классификация (Вид)_вид_89,Классификация (Вид)_вид_9,Классификация (Вид)_вид_90,Классификация (Вид)_вид_91,Классификация (Вид)_вид_92,Классификация (Вид)_вид_93,Классификация (Вид)_вид_94,Классификация (Вид)_вид_95,Классификация (Вид)_вид_96,Классификация (Вид)_вид_97,Классификация (Вид)_вид_98,Классификация (Вид)_вид_99
i32,i8,i8,i16,i8,i64,i8,u32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
2019,4,14,93,3,10,1,52,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [34]:
X = df_fea.clone()
target_col = 'Статус_устранения'
y = df_model.select(target_col)

print("\nПризнаки (X):")
print(X.head())
print(f"\nРазмер X: {X.shape}")

print("\nЦелевая переменная (y):")
print(y.head())
print(f"\nРазмер y: {y.shape}")



Признаки (X):
shape: (5, 1_057)
┌──────┬────────────┬────────────┬───────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ Год  ┆ Номер_меся ┆ Номер_неде ┆ Номер_дня ┆ … ┆ Классифика ┆ Классифика ┆ Классифик ┆ Классифик │
│ ---  ┆ ца         ┆ ли         ┆ ---       ┆   ┆ ция (Вид)_ ┆ ция (Вид)_ ┆ ация (Вид ┆ ация (Вид │
│ i32  ┆ ---        ┆ ---        ┆ i16       ┆   ┆ вид_96     ┆ вид_97     ┆ )_вид_98  ┆ )_вид_99  │
│      ┆ i8         ┆ i8         ┆           ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│      ┆            ┆            ┆           ┆   ┆ u8         ┆ u8         ┆ u8        ┆ u8        │
╞══════╪════════════╪════════════╪═══════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 2019 ┆ 4          ┆ 14         ┆ 93        ┆ … ┆ 0          ┆ 0          ┆ 0         ┆ 0         │
│ 2019 ┆ 4          ┆ 14         ┆ 93        ┆ … ┆ 0          ┆ 0          ┆ 0         ┆ 0         │
│ 2019 ┆ 4          ┆ 14         ┆ 93        ┆ … ┆ 0      

In [38]:
y['Статус_устранения'].value_counts()

Статус_устранения,count
i32,u32
1,37691
-1,2
0,8298
